In [1]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.1/320.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 14.2 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import torch
from transformers import BertTokenizer,BertForSequenceClassification
from sklearn.metrics import root_mean_squared_error,classification_report,confusion_matrix

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def tokenization(sent1, sent2):
    encoded = tokenizer.encode_plus(
        sent1,sent2,
        add_special_tokens=True,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoded['input_ids']
    attention_masks = encoded['attention_mask']
    return input_ids, attention_masks

In [ ]:
import string
def remove_special_characters(text):
    translator=str.maketrans('','',string.punctuation)
    return text.translate(translator)

def lower_case(text):
    text=text.lower()
    return text

def remove_stopwords(text):
    words=text.split()
    stop_words=set(stopwords.words('english'))
    words=[word for word in words if word not in stop_words]
    return ' '.join(words)

def preprocess(text):
    text=remove_special_characters(text)
    text=lower_case(text)
    text=remove_stopwords(text)
    return text

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model2=BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/model21').to(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gradio as gr
def predict_score_and_accuracy(text1,text2,num):
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        text1=preprocess(text1)
        text2=preprocess(text2)
        input_ids,attention_mask=tokenization(text1,text2)
        with torch.no_grad():
            model2.to(device)
            input_ids=input_ids.to(device)
            attention_masks=attention_mask.to(device)
            outputs=model2(input_ids,attention_masks)
            logits=outputs.logits
            mark_obtained=torch.sigmoid(logits)
            accuracy=torch.argmax(logits,dim=1).item()
            if accuracy==0:
                avg_marks=torch.min(mark_obtained).item()
                avg_marks=round(avg_marks*num,0)
                if avg_marks==num or avg_marks==num-1:
                  accuracy='High'
                elif avg_marks>(num/2) and avg_marks<num-1:
                  accuracy='Medium'
                elif avg_marks<(num/2):
                  accuracy='Low'
            elif accuracy==1:
                avg_marks=torch.mean(mark_obtained).item()
                avg_marks=round(avg_marks*num,0)
                if avg_marks==num or avg_marks==num-1:
                  accuracy='High'
                elif avg_marks>(num/2) and avg_marks<num-1:
                  accuracy='Medium'
                elif avg_marks<(num/2):
                  accuracy='Low'
            elif accuracy==2:
                avg_marks=torch.max(mark_obtained).item()
                avg_marks=round(avg_marks*num,0)
                if avg_marks==num or avg_marks==num-1:
                  accuracy='High'
                elif avg_marks>(num/2) and avg_marks<num-1:
                  accuracy='Medium'
                elif avg_marks<(num/2):
                  accuracy='Low'
        return accuracy,avg_marks

iface = gr.Interface(
    fn=predict_score_and_accuracy,
    inputs=[gr.Textbox(label='actual answer'),gr.Textbox(label='student answer'),gr.Number(label='Maximum Marks')],
    outputs=[gr.Textbox(label='Accuracy'),gr.Number(label='Marks Obtained')]
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a4298ae4745ccaa69.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
